# Report assets
Figures and GIFs for the report (hoyGarageRoofHydrant1).

In [1]:
import glob, json, math, os, shutil, subprocess, sys, tempfile
from datetime import datetime
from pathlib import Path

import imageio.v2 as imageio
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, MultipleLocator
from PIL import Image, ImageOps

HERE = Path.cwd()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / "scripts").exists())
os.chdir(ROOT)

DS = "hoyGarageRoofHydrant1"
RAW = Path(f"data/raw/{DS}")
PANO = Path(f"data/registered/{DS}/images/pano_aligned")
TRAINED = Path(f"data/trained/{DS}")
OUT = HERE.relative_to(ROOT)
REPORT = Path(OUT.name.removesuffix("_assets") + ".md")
CACHE = TRAINED / "frame_cache"
EVAL = TRAINED / "eval_cache"

SAMPLES = ["2023-08-16T13:59:35", "2023-12-03T12:21:05", "2024-04-19T14:47:39", "2024-10-07T10:07:07"]
DETAIL = ["2023-12-03T12:21:05", "2024-10-07T10:07:07"]
CKPT = 29999
STEPS = [4999, 9999, 19999, 29999]
RUN_EVAL = False

# gif: fps, width, keep every n-th frame
GIF = {"raw_vs_aligned": (6, 560, 4), "aligned_panos": (6, 480, 5),
       "progress_2x2": (5, 520, 1), "render_albedo_shading": (6, 780, 1)}

# render_traj uses fx from a 120 deg fov, training uses fx = H/2 -> crop to the trained canvas
ch = int(2 * 1080 / (2 * math.tan(math.radians(60))) * 0.96) // 2 * 2
cw = int(ch * 4 / 3) // 2 * 2
CROP = (cw, ch, (1920 - cw) // 2, (1080 - ch) // 2)

FMT = "%Y-%m-%dT%H:%M:%S"
STEMS = sorted(Path(p).stem for p in glob.glob(str(PANO / "*.png")))
DAYS = sorted({datetime.strptime(s, FMT).date() for s in STEMS})
for d in ["raw", "pano", "video"]:
    (CACHE / d).mkdir(parents=True, exist_ok=True)

In [2]:
def safe(s):
    return s.replace(":", "-")

def reset(d):
    d = OUT / d
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True)
    return d

def raw(s):
    return Path(glob.glob(str(RAW / "primary" / f"{s}.*"))[0])

def pano(s):
    p = Image.open(PANO / f"{s}.png").convert("RGBA")
    im = Image.new("RGB", p.size)
    im.paste(p, mask=p.getchannel("A"))
    return im

def save(im, path):
    if im.width > 1440:
        im = im.resize((1440, round(im.height * 1440 / im.width)), Image.LANCZOS)
    im.save(path, quality=85)

def box(im):
    im = im.copy(); im.thumbnail((640, 480), Image.LANCZOS)
    out = Image.new("RGB", (640, 480))
    out.paste(im, ((640 - im.width) // 2, (480 - im.height) // 2))
    return out

def crop(im):
    w, h, x, y = CROP
    return im.resize((1920, 1080), Image.LANCZOS).crop((x, y, x + w, y + h))

def ff(*a):
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", *map(str, a)], check=True)

X264 = ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "23"]
CROPF = "scale=1920:1080,crop={}:{}:{}:{}".format(*CROP)

def gif(src, dst, fps, w, n):
    sel = f"select='not(mod(n\\,{n}))',setpts=N/({fps}*TB)," if n > 1 else ""
    ff("-i", src, "-vf", f"{sel}fps={fps},scale={w}:-1:flags=lanczos,split[a][b];"
       "[a]palettegen=max_colors=64:stats_mode=diff[p];[b][p]paletteuse=dither=bayer:bayer_scale=4", dst)

def label(t, s):
    return (f"drawtext=fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf:text='{t}':"
            f"x=12:y=10:fontsize={s}:fontcolor=white:box=1:boxcolor=black:boxborderw=6")

def k(x, _=None):
    return "0" if x == 0 else f"{x / 1000:g}k"

BLUE, GREY, LINE, BG = "#2a78d6", "#52514e", "#e6e5e1", "#fcfcfb"
plt.rcParams.update({"font.size": 11, "axes.edgecolor": LINE, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": LINE,
                     "figure.facecolor": BG, "axes.facecolor": BG, "axes.titleweight": "bold",
                     "axes.titlesize": 12, "axes.titlelocation": "left", "figure.dpi": 150})

## raw + registration

In [3]:
d = reset("01_raw") / "primary"; d.mkdir()
for s in SAMPLES:
    shutil.copy2(raw(s), d / (safe(s) + raw(s).suffix))

reg = reset("02_registration")
(reg / "pano_aligned").mkdir()
for s in SAMPLES:
    save(pano(s), reg / "pano_aligned" / f"{safe(s)}.jpg")

if len(list((CACHE / "raw").glob("*.jpg"))) != len(STEMS):
    for i, s in enumerate(STEMS):
        box(ImageOps.exif_transpose(Image.open(raw(s))).convert("RGB")).save(CACHE / "raw" / f"{i:04d}.jpg", quality=90)
        box(pano(s)).save(CACHE / "pano" / f"{i:04d}.jpg", quality=90)

v = CACHE / "video"
ff("-framerate", 12, "-i", CACHE / "pano" / "%04d.jpg", *X264, v / "aligned_panos.mp4")
ff("-framerate", 12, "-i", CACHE / "raw" / "%04d.jpg", "-framerate", 12, "-i", CACHE / "pano" / "%04d.jpg",
   "-filter_complex", "[0][1]hstack=2", *X264, v / "raw_vs_aligned.mp4")
for n in ["aligned_panos", "raw_vs_aligned"]:
    gif(v / f"{n}.mp4", reg / f"{n}.gif", *GIF[n])

## training

In [4]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

tr = reset("03_training")
sc = {}
for ev in sorted(glob.glob(str(TRAINED / "tb" / "events.*"))):
    ea = EventAccumulator(ev, size_guidance={"scalars": 0}); ea.Reload()
    for tag in ea.Tags()["scalars"]:
        sc.setdefault(tag, {}).update({e.step: e.value for e in ea.Scalars(tag)})

def curve(tag, title, name, scale=1, avg=True):
    x = np.array(sorted(sc[tag])); y = np.array([sc[tag][i] for i in x]) * scale
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    if avg:
        ax.plot(x, y, color=BLUE, alpha=0.25, lw=1)
        m = np.convolve(y, np.ones(10) / 10, "valid")
        ax.plot(x[len(x) - len(m):], m, color=BLUE, lw=2)
    else:
        ax.plot(x, y, color=BLUE, lw=2)
    ax.xaxis.set_major_locator(MultipleLocator(5000)); ax.xaxis.set_major_formatter(FuncFormatter(k))
    ax.set_title(title); ax.set_xlabel("step")
    fig.tight_layout(); fig.savefig(tr / name); plt.close(fig)

curve("train/loss", "Total loss (0.8·L1 + 0.2·D-SSIM)", "loss_total.png")
curve("train/l1loss", "L1 loss", "loss_l1.png")
curve("train/ssimloss", "1 − SSIM", "loss_ssim.png")
curve("train/num_GS", "Albedo Gaussians (millions)", "num_gaussians.png", 1e-6, False)

fc = ";".join(f"[{i}]{CROPF},scale=640:-2,{label(k(s + 1), 28)}[v{i}]" for i, s in enumerate(STEPS))
ff(*sum([["-i", TRAINED / "videos" / f"traj_{s}.mp4"] for s in STEPS], []),
   "-filter_complex", fc + ";[v0][v1][v2][v3]xstack=inputs=4:layout=0_0|w0_0|0_h0|w0_h0", *X264, v / "progress_2x2.mp4")
gif(v / "progress_2x2.mp4", tr / "progress_2x2.gif", *GIF["progress_2x2"])

## rendering

In [5]:
rd = OUT / "04_rendering"
rd.mkdir(exist_ok=True)
traj = reset("04_rendering/trajectory")
vids = {c: TRAINED / "videos" / f"traj_{CKPT}{x}.mp4" for c, x in [("render", ""), ("albedo", "_albedo"), ("shading", "_shading")]}

for c, path in vids.items():
    r = imageio.get_reader(path); n = r.count_frames()
    for s in DETAIL:
        t = DAYS.index(datetime.strptime(s, FMT).date()) / (len(DAYS) - 1)
        crop(Image.fromarray(r.get_data(round(t * (n - 1))))).save(traj / f"{s[:10]}_{c}.jpg", quality=85)
    r.close()

fc = ";".join(f"[{i}]{CROPF},scale=600:-2,{label(t, 24)}[v{i}]"
              for i, t in enumerate(["render = albedo x shading", "albedo", "shading"]))
ff(*sum([["-i", p] for p in vids.values()], []), "-filter_complex", fc + ";[v0][v1][v2]hstack=3",
   *X264, v / "render_albedo_shading.mp4")
gif(v / "render_albedo_shading.mp4", rd / "render_albedo_shading.gif", *GIF["render_albedo_shading"])

## eval (gpu)
All panos rendered at their own time and sun angle, scored inside the alpha mask.

In [6]:
if RUN_EVAL:
    import torch
    sys.path.insert(0, "time-splatting")
    import train as ts
    from fused_ssim import fused_ssim
    from gsplat.strategy import DefaultStrategy
    from options import TimeSplattingConfig
    from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

    EVAL.mkdir(parents=True, exist_ok=True)
    cfg = TimeSplattingConfig(strategy=DefaultStrategy(reset_every=100000), shading_strategy=DefaultStrategy(reset_every=100000),
                              data_dir=str(PANO), result_dir=tempfile.mkdtemp(), disable_viewer=True)
    run = ts.Runner(cfg)
    ck = torch.load(TRAINED / "ckpts" / f"ckpt_{CKPT}.pt", map_location="cuda", weights_only=True)
    for key in run.splats.keys(): run.splats[key].data = ck["splats"][key]
    for key in run.shading_splats.keys(): run.shading_splats[key].data = ck["shading_splats"][key]
    lpips = LearnedPerceptualImagePatchSimilarity(net_type="alex", normalize=True).cuda()
    to_img = lambda x: Image.fromarray((torch.clamp(x, 0, 1)[0].cpu().numpy() * 255).astype(np.uint8))
    half = lambda x: torch.nn.functional.interpolate(x, scale_factor=0.5)

    rows = []
    with torch.no_grad():
        for i, path in enumerate(run.trainset.image_paths):
            d = run.trainset[i]; name = Path(path).stem
            gt, a = d["image"].cuda()[None] / 255, d["alpha"].cuda()[None] / 255
            H, W = gt.shape[1:3]
            kw = dict(camtoworlds=d["camtoworld"].cuda()[None], Ks=d["K"].cuda()[None], width=W, height=H,
                      near_plane=0.01, far_plane=100, render_mode="RGB")
            t = torch.tensor([d["time"]]).float().cuda()
            alb, _, _ = run.rasterize_splats(run.splats, t, backgrounds=torch.ones(1, 3, device="cuda"), **kw)
            sh, _, _ = run.rasterize_splats(run.shading_splats, torch.cat([t, d["sun_angle"].cuda()]),
                                            backgrounds=torch.ones(1, 1, device="cuda"), **kw)
            rgb = torch.clamp(alb * sh, 0, 1)
            m = a[..., 0] > 0.5
            p, g = (rgb * a).permute(0, 3, 1, 2), (gt * a).permute(0, 3, 1, 2)
            rows.append({"frame": name, "psnr": (-10 * torch.log10(((rgb - gt) ** 2)[m].mean())).item(),
                         "ssim": fused_ssim(p, g, padding="valid").item(), "lpips": lpips(half(p), half(g)).item(),
                         "coverage": m.float().mean().item()})
            for c, im in [("input", gt * a), ("render", rgb), ("albedo", alb), ("shading", sh.repeat(1, 1, 1, 3))]:
                save(to_img(im), EVAL / f"{name}_{c}.jpg")

    summary = {key: float(np.mean([r[key] for r in rows])) for key in ["psnr", "ssim", "lpips"]}
    json.dump({"summary": summary, "frames": rows}, open(EVAL / "metrics.json", "w"), indent=1)
    del run, ck; torch.cuda.empty_cache()

rec = OUT / "04_rendering" / "reconstruction"
shutil.rmtree(rec, ignore_errors=True); rec.mkdir()
for s in DETAIL:
    for c in ["input", "render", "albedo", "shading"]:
        Image.open(EVAL / f"{s}_{c}.jpg").save(rec / f"{safe(s)}_{c}.jpg", quality=85)

metrics = json.load(open(EVAL / "metrics.json"))
print({key: round(val, 3) for key, val in metrics["summary"].items()}, len(metrics["frames"]), "frames")

{'psnr': 19.163, 'ssim': 0.758, 'lpips': 0.352, 'n': 406} 406 frames


## metric charts

In [7]:
rm = reset("05_remarks")
frames = metrics["frames"]
dates = [datetime.strptime(r["frame"], FMT) for r in frames]

def per_frame(key, ylabel, unit, fmt):
    vals = [r[key] for r in frames]; mu = metrics["summary"][key]
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(dates, vals, color=BLUE, lw=0.8, alpha=0.3)
    ax.scatter(dates, vals, color=BLUE, s=22, edgecolors=BG, linewidths=0.8, zorder=3)
    ax.xaxis.set_major_locator(mdates.MonthLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.axhline(mu, color=GREY, lw=1, ls="--")
    ax.text(1.0, mu, f" mean\n {mu:{fmt}}{unit}", transform=ax.get_yaxis_transform(), color=GREY, va="center", fontsize=9)
    pad = (max(vals) - min(vals)) * 0.08
    ax.set_ylim(min(vals) - pad, max(vals) + pad); ax.set_ylabel(ylabel)
    ax.set_title(f"Masked {ylabel} per frame, render vs. input ({len(frames)} panos)", pad=12)
    plt.setp(ax.get_xticklabels(), rotation=60, ha="right", fontsize=9)
    fig.tight_layout(); fig.savefig(rm / f"{key}_per_frame.png"); plt.close(fig)

per_frame("psnr", "PSNR (dB)", " dB", ".1f")
per_frame("ssim", "SSIM", "", ".3f")

## check

In [8]:
import re
files = {str(f) for f in OUT.rglob("*") if f.is_file() and f.suffix != ".ipynb"}
links = {a or b for a, b in re.findall(r'\]\(([^)#\s]+)\)|src="([^"]+)"', REPORT.read_text())}
links = {l for l in links if l.startswith(str(OUT))}
print(f"{sum(Path(f).stat().st_size for f in files) / 1e6:.1f} MB, {len(files)} files")
print("missing:", sorted(links - files) or "none", "| unused:", sorted(files - links) or "none")

21.6 MB, 32 files
missing: none | unused: none
